In [3]:

# 1. Import required libraries
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# 2. Load Breast Cancer dataset
data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset Shape:", X.shape)
print("Target Shape:", y.shape)

# 3. Split dataset into Training, Validation and Testing sets

# First split: 60% training, 40% remaining
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# Second split: 20% validation, 20% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("\nTraining Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)
print("Testing Shape:", X_test.shape)


# 4. Train Logistic Regression as baseline

baseline = LogisticRegression(max_iter=10000)

baseline.fit(X_train, y_train)

# Predict validation data
baseline_pred = baseline.predict(X_val)

baseline_accuracy = accuracy_score(y_val, baseline_pred)

print("\nBaseline Logistic Regression Accuracy:",
      baseline_accuracy)


# 5. Use 5-Fold Cross Validation

cv_scores = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("\n5-Fold Cross Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())


# 6. Create Pipeline with StandardScaler and Logistic Regression

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=10000))
])

# Train pipeline
pipeline.fit(X_train, y_train)

# Validation prediction
pipeline_pred = pipeline.predict(X_val)

pipeline_accuracy = accuracy_score(y_val, pipeline_pred)

print("\nPipeline Accuracy:", pipeline_accuracy)


# 7. Hyperparameter tuning using GridSearchCV

param_grid = {
    "logistic__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train, y_train)

print("\nBest Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)


# 8. Evaluate tuned model on validation data

best_model = grid_search.best_estimator_

tuned_pred = best_model.predict(X_val)

tuned_accuracy = accuracy_score(y_val, tuned_pred)

print("\nTuned Model Validation Accuracy:", tuned_accuracy)


# 9. Compare all models

print("\n----- MODEL COMPARISON -----")

print("Baseline Accuracy:", baseline_accuracy)
print("Pipeline Accuracy:", pipeline_accuracy)
print("Tuned Model Accuracy:", tuned_accuracy)


# 10. Final evaluation on Test dataset

test_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_pred)

print("\nFinal Test Accuracy:", test_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_test,
    test_pred,
    target_names=data.target_names
))


Dataset Shape: (569, 30)
Target Shape: (569,)

Training Shape: (341, 30)
Validation Shape: (114, 30)
Testing Shape: (114, 30)

Baseline Logistic Regression Accuracy: 0.9649122807017544

5-Fold Cross Validation Scores: [0.92753623 0.94117647 0.92647059 0.98529412 0.91176471]
Mean CV Accuracy: 0.9384484228473999

Pipeline Accuracy: 0.9912280701754386

Best Parameters: {'logistic__C': 0.1}
Best CV Accuracy: 0.9736146632566071

Tuned Model Validation Accuracy: 0.9824561403508771

----- MODEL COMPARISON -----
Baseline Accuracy: 0.9649122807017544
Pipeline Accuracy: 0.9912280701754386
Tuned Model Accuracy: 0.9824561403508771

Final Test Accuracy: 0.9824561403508771

Classification Report:
              precision    recall  f1-score   support

   malignant       1.00      0.95      0.98        43
      benign       0.97      1.00      0.99        71

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98   